# Data observability

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/data-observability.ipynb)

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

Data observability is the practice of monitoring, alerting, and diagnosing the health of data and data pipelines. It focuses on whether datasets arrive on time, with the expected volume and values, and whether downstream consumers can trust them. Application monitoring tells you the job succeeded; data observability tells you the data is right.

The discipline is organized around five pillars: **freshness**, **volume**, **schema**, **quality**, and **lineage**. A useful framing for interviews is that observability is *metrics + logs + lineage + tests, joined by a common dataset and run identifier*.

## What is data observability?

The minimum viable implementation is a control table that every run writes to, so any run can be reconstructed months later:

```sql
CREATE TABLE IF NOT EXISTS platform.pipeline_runs (
    run_id        STRING,
    pipeline      STRING,
    dataset       STRING,
    logical_date  DATE,
    started_at    TIMESTAMP,
    finished_at   TIMESTAMP,
    status        STRING,        -- success | failed | quarantined
    rows_read     BIGINT,
    rows_written  BIGINT,
    bytes_scanned BIGINT,
    error_message STRING,
    code_version  STRING
);

-- Freshness and volume for the latest run of each dataset
SELECT
    dataset,
    MAX(finished_at) AS last_success,
    SUM(rows_written) AS rows_today
FROM platform.pipeline_runs
WHERE logical_date = CURRENT_DATE() AND status = 'success'
GROUP BY dataset
ORDER BY last_success ASC;
```

The same control table, runnable on DuckDB:

In [ ]:
# The control table in DuckDB: every run writes one row, and freshness/volume checks read it.
from datetime import datetime

import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE pipeline_runs (
        run_id VARCHAR, dataset VARCHAR, logical_date DATE, finished_at TIMESTAMP,
        status VARCHAR, rows_written BIGINT
    );
    INSERT INTO pipeline_runs VALUES
        ('r1', 'curated.orders',   DATE '2026-10-03', TIMESTAMP '2026-10-04 05:40', 'success', 1000),
        ('r2', 'curated.orders',   DATE '2026-10-04', TIMESTAMP '2026-10-05 05:35', 'success', 1040),
        ('r3', 'curated.payments', DATE '2026-10-03', TIMESTAMP '2026-10-04 05:50', 'success',  990),
        ('r4', 'curated.payments', DATE '2026-10-04', TIMESTAMP '2026-10-05 05:55', 'failed',     0),
        ('r5', 'curated.refunds',  DATE '2026-10-04', TIMESTAMP '2026-10-05 06:10', 'success',    3);
""")

NOW = datetime(2026, 10, 5, 8, 0)
SLA_MINUTES = 6 * 60

health = con.execute("""
    WITH latest AS (
        SELECT dataset, max(finished_at) AS last_success,
               arg_max(rows_written, finished_at) AS last_rows,
               median(rows_written) AS median_rows
        FROM pipeline_runs WHERE status = 'success'
        GROUP BY dataset
    )
    SELECT dataset,
           date_diff('minute', last_success, CAST(? AS TIMESTAMP)) AS minutes_stale,
           last_rows, median_rows
    FROM latest ORDER BY dataset
""", [NOW]).fetchall()

for dataset, stale, rows, median_rows in health:
    flags = [f for f, bad in (("STALE", stale > SLA_MINUTES), ("LOW_VOLUME", rows < 0.5 * median_rows)) if bad]
    print(f"{dataset:18s} stale={stale:4d}min rows={rows:5d} {' '.join(flags) or 'ok'}")

stale = {d for d, m, _, _ in health if m > SLA_MINUTES}
assert stale == {"curated.payments"}           # last success was the day before: the failed run shows up

Commercial tools (Monte Carlo, Bigeye, Soda, Elementary) automate the same checks; open-source stacks do it with dbt tests, Great Expectations, OpenLineage, and Prometheus. The tool matters less than having an owner, an SLA, and a runbook per critical dataset.

## What signals should you monitor for a data pipeline?

Monitor freshness, volume, schema, quality, latency, cost, and failure rate. Infrastructure metrics are useful, but data-specific signals catch silent failures that compute metrics miss.

| Signal | Question it answers | Typical source |
|---|---|---|
| Freshness | Is the newest partition recent enough? | max(event_ts) vs SLA |
| Volume | Did row counts stay in range? | count(*) per partition |
| Schema | Did columns or types change? | information_schema diff |
| Quality | Are values valid and unique? | dbt tests, GE, SQL checks |
| Latency | How long from event to availability? | event_ts vs publish_ts |
| Reliability | Did the job fail, retry, or run late? | orchestrator metadata |

Expose these as time series so you can alert on trend, not just pass/fail:

```yaml
# prometheus/rules/data_pipeline.yml
groups:
  - name: data-pipeline
    rules:
      - alert: DatasetStale
        expr: time() - pipeline_last_success_timestamp_seconds > 7200
        for: 10m
        labels: {severity: page, team: data-platform}
        annotations:
          summary: "{{ $labels.dataset }} is more than 2h stale"
          runbook_url: "https://runbooks.acme.io/dataset-stale"
```

Emit the gauges from the pipeline itself or from an exporter that queries the control table, so alerts do not depend on someone reading a dashboard.

For each signal define a threshold, an owner, and a query that can be run during an incident. How to write the checks themselves (freshness lag per partition, volume against a weekday baseline, robust anomaly detection, quarantine) is covered in [Data quality](data-quality.ipynb); observability is where their results become time series, alerts, and routed pages.

Alert on **rate of change**, not absolute values: a 30% volume drop on a holiday means something different than on a normal Tuesday, so baseline by weekday and compare against a trailing window.

## How do you reduce noisy data alerts?

Make alerts actionable and severity-based. Use thresholds with context, anomaly detection where useful, and alert only when a user-facing SLA or critical data contract is at risk. Non-critical checks can go to dashboards or daily reports.

```text
page:      gold revenue table is stale past 8 AM SLA
ticket:    staging null rate increased but downstream mart is unaffected
dashboard: row count changed within expected seasonality
```

Map every rule to one of three tiers and route it accordingly:

+ **page** - consumer-visible, SLA-breaching, one owner, runbook required.
+ **ticket** - real defect with a slower clock, no immediate user impact.
+ **dashboard-only** - informational; reviewed weekly, never wakes anyone.

In Prometheus, severity plus inhibition suppresses the downstream blast radius so one root cause produces one page:

```yaml
# prometheus/rules/inhibit.yml
inhibit_rules:
  - source_matchers: [severity="page", alertname="SourceFeedMissing"]
    target_matchers: [severity="page", alertname=~"DatasetStale|VolumeAnomaly"]
    equal: [domain]
```

A check that has never fired, or always fires and is ignored, should be deleted or downgraded. Noisy alerts train teams to ignore the system, so alert design is part of reliability.

Rules that work: one alert, one action; alert on symptoms rather than every cause; use `for:` durations so blips do not page; group and inhibit so one root cause produces one notification; baseline by season with robust statistics (median, MAD); route non-paging tiers to a queue; and review alert quality monthly.

A multi-signal rule fires only when freshness **and** volume are both wrong, which is far less noisy than either alone:

```yaml
# prometheus/rules/revenue_mart.yml
groups:
  - name: revenue-mart
    rules:
      - alert: RevenueMartDegraded
        expr: |
          (time() - mart_last_success_timestamp_seconds{dataset="marts.revenue_daily"} > 10800)
          and
          (abs(mart_rows_written_total{dataset="marts.revenue_daily"} - mart_rows_written_7d_median)
             > 0.4 * mart_rows_written_7d_median)
        for: 15m
        labels: {severity: page, team: revenue-data}
        annotations:
          summary: "revenue_daily is stale AND volume is off baseline"
          runbook_url: "https://runbooks.acme.io/revenue-mart-degraded"
```

Airflow-side alerting should distinguish **task failure** from an **SLA miss**, because a DAG can finish successfully while being hours late. Assign every critical dataset an owning team and keep the paging tier small; a platform where every dataset pages the same channel is one where nobody trusts the channel.

## How is data observability different from application observability?

Application observability focuses on service health (latency, error rates, traces) for request/response systems. Data observability focuses on dataset health and pipeline behavior over time (freshness, completeness, distribution shifts). Data issues can be silent, delayed, and cumulative, so you need different signals and baselines.

| Dimension | Application observability | Data observability |
|---|---|---|
| Unit of health | request / endpoint / pod | dataset / partition / column |
| Primary signals | latency, error rate, saturation | freshness, volume, schema, drift |
| Failure shape | usually loud and immediate | often silent until a metric drifts |
| Time horizon | seconds to minutes | hours to weeks (batch, seasonality) |
| "Correct" means | 200 OK, no exception | matches source and business rule |
| Tooling | Prometheus, Grafana, OTel | dbt tests, GE, Soda, OpenLineage |

They share plumbing: OpenTelemetry spans can carry the dataset and run id so one trace covers "task started -> file read -> table written":

```python
# Emitting a data-aware span from a pipeline task
from opentelemetry import trace

tracer = trace.get_tracer("acme.pipelines")

def load(ds: str, path: str) -> int:
    with tracer.start_as_current_span("load_orders") as span:
        span.set_attribute("data.dataset", "curated.orders")
        span.set_attribute("data.partition", ds)
        rows = write_parquet(path, ds)
        span.set_attribute("data.rows_written", rows)
        return rows
```

The key interview point: **a green pipeline run is not evidence that the data is right.** You need dataset-level assertions on top of task-level success.

## How do you define and measure an end-to-end SLA for data?

You define the SLA from source event time to availability in the final dataset or BI metric. Measure it with timestamps at each stage (ingest, transform, publish) and compute percentiles (p50/p95) as well as breach counts. SLAs should reflect consumer needs and include ownership and escalation paths.

```text
event_time -> ingested_at -> transformed_at -> published_at -> dashboard_refreshed_at
             (ingest lag)    (transform lag)   (publish lag)
```

```sql
WITH staged AS (
    SELECT
        event_time,
        published_at,
        TIMESTAMP_DIFF(ingested_at, event_time, SECOND)   AS ingest_lag_s,
        TIMESTAMP_DIFF(published_at, ingested_at, SECOND) AS transform_lag_s,
        TIMESTAMP_DIFF(published_at, event_time, SECOND)  AS e2e_lag_s
    FROM curated.orders
    WHERE order_date >= CURRENT_DATE() - 30
)
SELECT
    DATE(published_at) AS publish_date,
    APPROX_QUANTILES(e2e_lag_s, 100)[OFFSET(50)] AS p50_s,
    APPROX_QUANTILES(e2e_lag_s, 100)[OFFSET(95)] AS p95_s,
    COUNTIF(e2e_lag_s > 4 * 3600) AS breaches_over_4h,
    COUNT(*) AS total_rows
FROM staged
GROUP BY publish_date
ORDER BY publish_date DESC;
```

Express the SLA as a contract with an owner, for example "99% of `curated.orders` rows with `event_time` before 23:59 UTC are queryable by 06:00 UTC, owned by the Orders Data team." Prefer a freshness SLO with an **error budget** to a hard promise, and measure freshness per partition, because a table can look fresh by max timestamp while one region partition silently stopped loading.

## What should you log for each pipeline run?

At minimum:

+ inputs (source tables/partitions, offsets, watermarks)
+ outputs (target tables/partitions, row counts written)
+ runtime and resource usage
+ warnings/errors and retry attempts
+ data quality results summary

This makes runs reproducible and speeds up debugging. Log structured JSON with a stable run id so logs can be joined to the control table and to traces:

In [ ]:
# Structured JSON logs with a stable run_id, captured here in memory so the cell is self-contained.
import io
import json
import logging
import uuid

stream = io.StringIO()
log = logging.getLogger("orders")
log.handlers = [logging.StreamHandler(stream)]
log.setLevel(logging.INFO)
log.propagate = False


def transform_and_write(source: str, target: str) -> int:
    return 1_040                                  # stand-in for the real transform


def run(ds: str, source: str, target: str) -> str:
    run_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{target}/{ds}"))   # deterministic per partition
    base = {"run_id": run_id, "dataset": target, "partition": ds, "code_version": "orders@2.4.1"}
    log.info(json.dumps({"event": "run_start", "source": source, **base}))
    rows = transform_and_write(source, target)
    log.info(json.dumps({"event": "run_end", "rows_written": rows, "status": "success", **base}))
    return run_id


run_id = run("2026-10-04", "raw.orders", "curated.orders")
events = [json.loads(line) for line in stream.getvalue().splitlines()]
for event in events:
    print(event)
assert [e["event"] for e in events] == ["run_start", "run_end"]
assert {e["run_id"] for e in events} == {run_id}  # joinable to the control table and traces

For Airflow, enable remote logging so lines carry DAG, task, run id, and try number:

```yaml
# airflow.cfg (excerpt)
[logging]
remote_logging = True
remote_log_conn_id = s3_logs
log_format = [%(asctime)s] {%(filename)s:%(lineno)d} %(levelname)s - %(message)s
```

Never log secrets or full PII payloads, always include the partition/logical date, and log a **redacted sample of rejected records** rather than only a count.

## How do you detect silent failures?

Silent failures happen when the job succeeds but data is wrong (missing subset, wrong join, truncation). Task-level success cannot catch them; dataset-level assertions can. The strongest signal is a reconciliation invariant that must hold regardless of seasonality (orders and payments reconcile in count and total for the day), backed by a fan-out guard (`COUNT(*)` equals `COUNT(DISTINCT order_id)` after a join) and freshness, volume, and distribution checks. How to write those checks, including dbt `data_tests:` and reconciliation queries, is covered in [Data quality](data-quality.ipynb).

The observability job is to make sure the checks run on every build, their results land in the control table with the run id, and a failure on a critical dataset pages its owner. Also watch key business KPIs: if revenue drops while row counts stay flat, a silent filter or join bug is the likely cause.

## What is lineage and how does it help during incidents?

Lineage describes upstream/downstream dependencies between datasets, jobs, and dashboards. During incidents, lineage helps you assess blast radius (which reports are impacted) and prioritize fixes. It also helps prevent regressions by making dependencies explicit.

Three useful levels: **table-level** (blast radius), **column-level** (metric drift debugging), and **run-level** (which specific run wrote the bad partition, for scoping a backfill). OpenLineage emits lineage as a side effect of running jobs so it stays current:

```python
# Emit dataset lineage from a pipeline task
from openlineage.client import OpenLineageClient
from openlineage.client.run import Dataset, RunEvent, RunState

client = OpenLineageClient(url="http://marquez:5000")

def emit_complete(run_id: str, job: str, inputs: list[str], outputs: list[str]) -> None:
    client.emit(RunEvent(
        eventType=RunState.COMPLETE,
        eventTime="2026-10-04T06:12:00Z",
        run=run_id,
        job=job,
        inputs=[Dataset(namespace="s3://acme-raw", name=n) for n in inputs],
        outputs=[Dataset(namespace="snowflake://acme", name=n) for n in outputs],
    ))
```

The same walk, over a lineage graph built from those events:

In [ ]:
# Lineage as a graph: walk upstream from a broken dashboard, downstream from a bad table.
from collections import deque

EDGES = {                       # producer -> consumers (what OpenLineage events let you build)
    "raw.orders": ["stg_orders"],
    "raw.payments": ["stg_payments"],
    "stg_orders": ["fct_orders"],
    "stg_payments": ["fct_orders", "fct_payments"],
    "fct_orders": ["revenue_daily", "ltv_model"],
    "fct_payments": ["finance_dashboard"],
    "revenue_daily": ["exec_dashboard"],
}


def walk(start: str, graph: dict[str, list[str]]) -> list[str]:
    seen, queue = [], deque(graph.get(start, []))
    while queue:
        node = queue.popleft()
        if node not in seen:
            seen.append(node)
            queue.extend(graph.get(node, []))
    return seen


upstream_graph: dict[str, list[str]] = {}
for producer, consumers in EDGES.items():
    for consumer in consumers:
        upstream_graph.setdefault(consumer, []).append(producer)

print("exec_dashboard depends on:", walk("exec_dashboard", upstream_graph))
print("blast radius of stg_payments:", walk("stg_payments", EDGES))
assert "raw.payments" in walk("exec_dashboard", upstream_graph)
assert set(walk("stg_payments", EDGES)) == {
    "fct_orders", "fct_payments", "revenue_daily", "ltv_model", "finance_dashboard", "exec_dashboard"}

During an incident, walk lineage **upstream from the broken dashboard** until the data is correct, then **downstream from the bad table** to list every affected metric and owner. The second walk is the difference between a 30-minute and a 4-hour incident.

## How do you monitor and debug a broken metric in BI?

You start from the metric definition and trace dependencies to the underlying tables. Then check data freshness and recent changes (deploys, schema changes, upstream outages). Use reconciliation queries and sampling to locate where the metric diverged, then confirm whether it is a data issue or a definition change.

A repeatable order that avoids guesswork: confirm the report (dashboard vs extract vs warehouse), check freshness, check volume, bisect the transformation layer by layer, check recent deploys, then reconcile against an independent source.

```sql
-- Bisect: compute the metric at every layer and find the first divergence
SELECT 'raw.orders'    AS layer, COUNT(*) AS n, SUM(amount) AS revenue FROM raw.orders      WHERE order_date = DATE '2026-10-03'
UNION ALL
SELECT 'stg_orders'    AS layer, COUNT(*) AS n, SUM(amount) AS revenue FROM staging.stg_orders WHERE order_date = DATE '2026-10-03'
UNION ALL
SELECT 'fct_orders'    AS layer, COUNT(*) AS n, SUM(amount) AS revenue FROM curated.fct_orders WHERE order_date = DATE '2026-10-03'
UNION ALL
SELECT 'revenue_daily' AS layer, COUNT(*) AS n, SUM(revenue) AS revenue FROM marts.revenue_daily WHERE order_date = DATE '2026-10-03'
ORDER BY layer;
```

Then sample the rows that differ to see the shape of the problem:

```sql
-- Rows present in staging but missing from the mart
SELECT s.order_id, s.amount, s.status
FROM staging.stg_orders AS s
LEFT JOIN curated.fct_orders AS f USING (order_id)
WHERE s.order_date = DATE '2026-10-03' AND f.order_id IS NULL
LIMIT 100;
```

Both queries together, runnable on DuckDB:

In [ ]:
# Bisect a broken metric: compute it at every layer and find the first divergence.
con.execute("""
    CREATE TABLE raw_orders AS SELECT * FROM (VALUES
        (1, 10.0, 'paid'), (2, 20.0, 'paid'), (3, 30.0, 'PAID'), (4, 40.0, 'refunded')
    ) AS t(order_id, amount, status);
    CREATE TABLE stg_orders AS SELECT order_id, amount, lower(status) AS status FROM raw_orders;
    -- the bug: a case-sensitive filter added in the mart build
    CREATE TABLE fct_orders AS SELECT * FROM raw_orders WHERE status = 'paid';
""")
layers = con.execute("""
    SELECT 'raw_orders' AS layer, count(*), sum(amount) FROM raw_orders WHERE status ILIKE 'paid'
    UNION ALL SELECT 'stg_orders', count(*), sum(amount) FROM stg_orders WHERE status = 'paid'
    UNION ALL SELECT 'fct_orders', count(*), sum(amount) FROM fct_orders
""").fetchall()
first_bad = next(layer for (layer, n, _), (_, n_prev, _) in zip(layers[1:], layers) if n != n_prev)
missing = con.execute("""
    SELECT s.order_id FROM stg_orders s LEFT JOIN fct_orders f USING (order_id)
    WHERE s.status = 'paid' AND f.order_id IS NULL
""").fetchall()
print(layers, "-> first divergence at", first_bad, "missing", missing)
assert first_bad == "fct_orders" and missing == [(3,)]

Also compare the metric query in version control with what the BI tool currently executes; a dashboard-side filter or a changed semantic-layer join is a common cause that no data test catches.

## How do you monitor a backfill?

Backfill design (scoping affected partitions, idempotent rewrites, shadow tables, and atomic swaps) is covered in [Data system design](system-design.ipynb). From an observability standpoint, a backfill is a planned incident: annotate or freeze the affected dashboards, capture before/after volume and key metrics per partition so the change is auditable, watch freshness and volume while it runs, and record what ran in the control table. For finance or ML training data, require a dry run on one partition and a written expected metric delta before the full run.

## What is a runbook and what should it contain?

A runbook is a step-by-step guide for responding to an alert or incident. It should include impact assessment, common root causes, validation queries, mitigation steps, rollback/backfill steps, and escalation contacts. Good runbooks reduce time-to-recovery for on-call engineers.

A good runbook is short, specific, and executable:

```text
Alert: RevenueMartDegraded          Owner: revenue-data (page)      SLA: correct by 08:00
Impact:   exec revenue dashboard stale or wrong; finance close may slip
Check:    1. freshness/volume of raw.orders and raw.payments (control-table query)
          2. last 3 runs of revenue_daily in the orchestrator (failed? late? retried?)
          3. bisect revenue across raw -> stg -> fct -> mart for the logical date
Mitigate: annotate the dashboard; rerun the failed task; if source is late, wait - do not backfill
Backfill: airflow backfill create --dag-id revenue_daily --from-date D --to-date D --reprocess-behavior failed
Escalate: orders-platform on-call if raw.orders is missing; data-platform if the warehouse is down
```

Store runbooks next to the code in the repo and link them from the alert annotation, so they are reviewed in the same pull request as the pipeline. A runbook that lives only in a wiki rots within a quarter.

## What are common incident patterns in data platforms?

| Pattern | Signature | Fast detection |
|---|---|---|
| Late source feed | freshness climbs, volume drops | max(event_ts) vs SLA |
| Schema change | job fails at parse, or nulls appear | schema contract diff |
| Retry duplicates | row counts exceed baseline | count(*) <> count(distinct pk) |
| Join fan-out | totals inflated | fan-out guard query |
| Timezone bug | a day has ~half or ~double volume | volume by hour of day |
| Runaway query | bytes scanned or spend spikes | query history cost ranking |
| Silent filter | a segment disappears | distinct-value count by dimension |

A timezone bug is invisible to row-level checks, so profile volume across the hours of a day:

```sql
SELECT order_date, EXTRACT(HOUR FROM event_ts) AS hour_of_day, COUNT(*) AS n
FROM raw.orders
WHERE event_ts >= TIMESTAMP '2026-10-03 00:00:00'
  AND event_ts <  TIMESTAMP '2026-10-05 00:00:00'
GROUP BY order_date, hour_of_day
ORDER BY order_date, hour_of_day;
```

**Failure-mode note:** observability itself can fail. A freshness check that queries the same warehouse that is down quietly stops reporting and looks green. Monitor the monitors: alert when a check has not reported within its expected interval, and keep critical probes independent of the pipelines they observe.

## Real world use cases for Data Observability

### 1. E-commerce revenue mart with an 8 AM SLA

A retailer's executive revenue dashboard must be correct by 08:00 local time. The nightly ELT loads ~400M order events into Snowflake; the mart is 90 GB and feeds exec dashboards and finance.

```yaml
# dbt: freshness plus contract tests on the tier-1 source
sources:
  - name: raw
    database: RAW
    schema: PUBLIC
    loaded_at_field: ingested_at
    freshness:
      warn_after: {count: 30, period: minute}
      error_after: {count: 90, period: minute}
    tables:
      - name: orders
        columns:
          - name: order_id
            data_tests: [unique, not_null]
```

**Why this design:** one freshness SLO plus a join fan-out guard, because this company's revenue bugs historically came from double-counted refunds rather than missing rows. Priority is tied to the exec SLA; everything else is a ticket.

### 2. Fintech reconciliation between core ledger and warehouse

A payments company mirrors its ledger into Snowflake. Every day at 02:00 the warehouse totals must equal the ledger totals to the cent, or the regulatory close is blocked. Volume is ~120M transactions/day.

```sql
-- Reconciliation gate; a non-empty result fails the pipeline (fail closed)
SELECT
    SUM(t.amount_cents) - SUM(w.amount_cents) AS delta_cents
FROM ledger.transactions AS t
FULL OUTER JOIN warehouse.transactions AS w USING (txn_id)
WHERE DATE(t.settled_at) = DATE '2026-10-04'
   OR DATE(w.settled_at) = DATE '2026-10-04'
HAVING SUM(t.amount_cents) - SUM(w.amount_cents) <> 0;
```

**Why this design:** the ledger is an independent source of truth, so reconciliation is the only check that can prove correctness. The pipeline fails closed because a wrong regulatory number is worse than a late one.

### 3. Multi-tenant SaaS where one tenant breaks everyone

A B2B platform ingests events from 3,000 tenants into a shared lake (~2 TB/day). One tenant's malformed payload once broke the shared parse and stalled all tenants' dashboards.

```sql
-- Per-tenant freshness and volume so anomalies are attributed, not aggregated
SELECT
    tenant_id,
    MAX(event_ts) AS latest_event,
    COUNT(*) AS events_today,
    COUNT(*) * 1.0 / NULLIF(AVG(COUNT(*)) OVER (), 0) AS relative_volume
FROM raw.events
WHERE event_date = CURRENT_DATE() - 1
GROUP BY tenant_id
ORDER BY relative_volume ASC
LIMIT 50;
```

**Why this design:** checks are partitioned by `tenant_id` so an alert names the tenant, and bad payloads are quarantined per tenant rather than failing the shared DAG. This trades a little per-tenant latency for platform-wide availability.

### 4. ML feature store freshness as a first-class SLO

A recommendation system refreshes 600 features for 40M users every 6 hours. Stale features silently degrade click-through with no error, so freshness is the top metric.

In [ ]:
# Feature freshness probe, run after each materialization
from datetime import datetime, timedelta, timezone

SLA = timedelta(hours=6)


def check_feature_freshness(conn, feature_view: str, now: datetime | None = None) -> None:
    now = now or datetime.now(timezone.utc)
    row = conn.execute(f"SELECT MAX(event_ts) FROM {feature_view}").fetchone()
    latest = row[0]
    if latest is None or (now - latest) > SLA:
        raise RuntimeError(f"{feature_view} stale: latest={latest} now={now}")


con.execute("""
    CREATE TABLE user_features AS
    SELECT 'u' || i AS user_id, TIMESTAMPTZ '2026-10-05 02:00:00+00' AS event_ts FROM range(3) AS r(i)
""")
check_feature_freshness(con, "user_features", now=datetime(2026, 10, 5, 7, 0, tzinfo=timezone.utc))
try:
    check_feature_freshness(con, "user_features", now=datetime(2026, 10, 5, 9, 0, tzinfo=timezone.utc))
except RuntimeError as error:
    print("paged:", error)
else:
    raise AssertionError("a 7-hour-old feature view must breach the 6-hour SLA")

**Why this design:** freshness drives model quality, so it belongs in the same SLO framework as revenue data. Pairing it with a distribution-drift check catches the case where features are timely but shifted by a bad upstream join.

### Try it - Error budget arithmetic

A freshness SLO is a target plus an error budget, and the budget is just the allowed failure time in the window. Compute it once and you can answer "can we ship this?" without a spreadsheet.

In [ ]:
# Task: for a 99.9% freshness SLO over a 30-day window, compute the allowed
#   failure time and how much budget a 15-minute outage consumes.
# Hint: budget minutes = (1 - SLO) * window days * 24 * 60, and consumed is
#   spent minutes divided by that budget.
# Expected: 43.2 minutes of budget, 34.7% consumed, 28.2 minutes left.

SLO = 0.999
WINDOW_DAYS = 30

allowed_minutes = (1 - SLO) * WINDOW_DAYS * 24 * 60
spent_minutes = 15.0
remaining_minutes = allowed_minutes - spent_minutes

print(f"budget={allowed_minutes:.1f} min, spent={spent_minutes:.0f} min, "
      f"consumed={spent_minutes / allowed_minutes:.1%}, "
      f"remaining={remaining_minutes:.1f} min")

assert round(allowed_minutes, 2) == 43.2
assert round(spent_minutes / allowed_minutes, 4) == 0.3472
assert round(remaining_minutes, 1) == 28.2

### Try it - Percentiles and burn rate

Latency and freshness SLAs are read at p50 and p95, and burn rate says how fast you are spending the error budget. Both fall out of a few lines of arithmetic.

In [ ]:
# Task: compute p50 and p95 end-to-end lag with the nearest-rank method, then
#   the burn rate for a window against a 99.9% SLO.
# Hint: the p95 value sits at rank ceil(0.95 * n) in the sorted list, and burn
#   rate is the observed error rate divided by the budget error rate (0.001).
# Expected: p50=110s, p95=900s, and a 15x burn rate, past the 14.4x page.

import math


def percentile(sorted_values, pct):
    rank = math.ceil(pct / 100 * len(sorted_values))
    return sorted_values[max(rank - 1, 0)]


lags = sorted([120, 130, 90, 100, 900, 110, 95, 105, 115, 125])
p50 = percentile(lags, 50)
p95 = percentile(lags, 95)

total = 10_000
failed = 150                      # 1.5% of the window missed the freshness SLA
error_rate = failed / total
burn_rate = error_rate / (1 - 0.999)

print(f"p50={p50}s p95={p95}s error_rate={error_rate:.2%} burn_rate={burn_rate:.0f}x")
assert (p50, p95) == (110, 900)
assert round(burn_rate, 1) == 15.0
assert burn_rate > 14.4           # multiwindow burn-rate page threshold

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What are the five pillars of data observability, and what does each catch?

**A.** Freshness catches data that is late, volume catches missing or inflated rows, schema catches columns or types that changed, quality catches invalid or duplicated values, and lineage shows the blast radius of a failure. A green job run proves only that the task finished, not that the data is right.

**Q2.** How is data observability different from application observability?

**A.** Application observability watches request and service health such as latency, error rate, and saturation on a scale of seconds to minutes, where failures are usually loud. Data observability watches dataset and partition health such as freshness, volume, and drift over hours to weeks, where failures are often silent until a metric drifts.

**Q3.** What is an SLA as a contract, and why prefer an SLO with an error budget?

**A.** Express it from source event time to availability, for example "99% of `curated.orders` rows with `event_time` before 23:59 UTC are queryable by 06:00 UTC, owned by the Orders Data team." An SLO with an error budget states the same target in allowed failure time, which supports decisions like slowing a risky release instead of making a hard promise.

**Q4.** What is the strongest signal for a silent failure, and why?

**A.** A reconciliation invariant that must hold regardless of seasonality, such as orders and payments having equal counts and totals for the day, because it compares against an independent source of truth. Add a fan-out guard so `COUNT(*)` equals `COUNT(DISTINCT order_id)` after a join, which catches metric inflation before it reaches a dashboard.

**Q5.** How do you use lineage during an incident?

**A.** Walk upstream from the broken dashboard until the data is correct to find the first bad layer, then walk downstream from the bad table to list every affected metric, owner, and dashboard. The downstream walk is the difference between a 30-minute and a 4-hour incident because it scopes communication and backfills.

**Q6.** What makes an alert worth paging on?

**A.** It maps to one consumer-visible, SLA-breaching symptom with a single owner and a runbook. Prefer multi-signal rules such as freshness and volume both wrong, add `for:` durations so blips do not page, inhibit downstream alerts so one root cause produces one notification, and route everything else to a ticket or dashboard.

**Q7.** Why is a freshness check that queries the same warehouse a failure mode?

**A.** When the warehouse is down the check quietly stops reporting and can look green, so observability itself fails silently. Monitor the monitors by alerting when a check has not reported within its expected interval, and keep critical probes independent of the pipelines and systems they observe.